In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from typing import TypedDict
from dotenv import load_dotenv


In [2]:
load_dotenv()
model = ChatOpenAI()


In [3]:
class BlogState(TypedDict):
    title: str
    outline: str
    content:str

In [5]:
from typing import TypedDict

class BlogState(TypedDict):
    title: str
    outline: str

In [7]:
def create_outline(state: BlogState) -> BlogState:
    title = state['title']

    prompt = f'Generate an outline for a blog on the topic - {title}'

    outline = model.invoke(prompt)

    state['outline'] = outline.content

    return state

In [8]:
def create_blog(state: BlogState)-> BlogState:
    title = state['title']
    outline = state['outline']

    prompt = f"write a detaileed blog on the title - {title}using the following outline"
    model.invoke(prompt).content
    state['content']


In [10]:
graph = StateGraph(BlogState)

graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)

graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', END)

workflow = graph.compile()

In [13]:
class BlogState(TypedDict):
    title: str
    outline: str
    blog: str

In [14]:
def create_outline(state: BlogState) -> BlogState:

    title = state['title']

    prompt = f'Generate an outline for a blog on the topic - {title}'

    outline = model.invoke(prompt)

    state['outline'] = outline.content

    return state

In [15]:
def create_blog(state: BlogState) -> BlogState:

    outline = state['outline']

    prompt = f'Write a detailed blog using this outline:\n{outline}'

    blog = model.invoke(prompt)

    state['blog'] = blog.content

    return state

In [17]:
initial_state = {
    'title': 'rise of ai in india'
}

final_state = workflow.invoke(initial_state)

print(final_state)

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}